# OpenVLA Inference Baseline

Load OpenVLA 7B with 4-bit quantization for LIBERO evaluation. This is the comparison baseline against SmolVLA LoRA.

In [ ]:
# Cell 1: Install OpenVLA inference dependencies
!pip install -q transformers bitsandbytes>=0.43.0 accelerate>=0.26.0 timm pillow
!pip install -q lerobot[libero]

import os
os.environ["MUJOCO_GL"] = "egl"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Load OpenVLA 7B with 4-bit quantization
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig
import torch

processor = AutoProcessor.from_pretrained(
    "openvla/openvla-7b", trust_remote_code=True
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

vla = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    device_map="auto",
)

peak_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"OpenVLA loaded. Peak VRAM: {peak_mem:.2f} GB")

In [ ]:
# Cell 3: Test single-image inference
from PIL import Image
import numpy as np

# Create a dummy test image (replace with actual LIBERO frame later)
dummy_img = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))

instruction = "pick up the red block"
prompt = f"In: What action should the robot take to {instruction}?\nOut:"

inputs = processor(prompt, dummy_img).to(vla.device, dtype=torch.bfloat16)
action = vla.predict_action(**inputs, unnorm_key="bridge_orig", do_sample=False)

print(f"Predicted action: {action}")
print(f"Action shape: {action.shape}")
print("OpenVLA inference: OK")

In [ ]:
# Cell 4: Load LIBERO-finetuned OpenVLA for fair comparison
# These are officially fine-tuned checkpoints with known performance

OPENVLA_LIBERO_CHECKPOINTS = {
    "libero_object": "openvla/openvla-7b-finetuned-libero-object",  # 88.4%
    "libero_spatial": "openvla/openvla-7b-finetuned-libero-spatial",  # 84.7%
    "libero_goal": "openvla/openvla-7b-finetuned-libero-goal",  # 79.2%
}

# For fair comparison, we use the LIBERO-Object checkpoint
# (same task suite as our SmolVLA training)
print("Available OpenVLA LIBERO checkpoints:")
for suite, ckpt in OPENVLA_LIBERO_CHECKPOINTS.items():
    print(f"  {suite}: {ckpt}")

# NOTE: These checkpoints are full fine-tuned (not LoRA), so they show
# OpenVLA's ceiling performance. Our SmolVLA LoRA result being competitive
# with these would be a strong V1 success signal.